# Demonstration of Physical Compiler + Atom Layout

## Import dependencies (TODO: add more details here)

In [2]:
from typing import Any, Literal, TypeVar

import numpy as np
from bloqade.analysis.fidelity import FidelityAnalysis
from bloqade.types import Qubit
from kirin.dialects import ilist
from kirin.dialects.ilist import IList

from bloqade import squin
from bloqade.gemini.common.dialects import qubit
from bloqade.lanes.arch.gemini.logical import steane7_initialize
from bloqade.lanes.arch.gemini.physical import get_arch_spec
from bloqade.lanes.noise_model import generate_simple_noise_model
from bloqade.lanes.passes import ASAPPlacePass
from bloqade.lanes.pipeline import PhysicalPipeline
from bloqade.lanes.transform import MoveToSquinPhysical

kernel = squin.kernel.add(qubit)
kernel.run_pass = squin.kernel.run_pass

In [ ]:
# We program at the squin level, so internally, we simply represent our logical qubits as a list of Qubits.
LogicalQubit = ilist.IList[Qubit, Literal[7]]

# TODO: add an image of the qubit layout

In [ ]:
def steane_slot_allocator():
    """Generates a qubit allocator for logical qubits.
    Tries to allocate logical qubits into the architecture in an efficient way to
    make parallelism in the logical gagdets as easily as possible in the move compiler.

    """
    # Defines the valid words
    slot_words = ilist.IList([0, 4, 8, 12, 16, 2, 6, 10, 14, 20])

    # Defines the valid "slots" where we can place a logical qubit
    slots = IList(
        [
            IList([(0, word_id, site_id) for site_id in range(7)])
            for word_id in slot_words
        ]
    )

    # print(f"slots: {slots}")

    @kernel
    def qalloc_slot(
        slot_index: int, theta: float, phi: float, lam: float
    ) -> LogicalQubit:
        """
        Allocates a steane qubit at a particular word "slot", using theta, phi, lam as parameters to the U3 gate.
        """
        def allocate_at(address: tuple[int, int, int]):
            return qubit.new_at(address[0], address[1], address[2])

        addresses = slots[slot_index]

        reg = ilist.map(allocate_at, addresses)

        steane7_initialize(theta, phi, lam, reg)

        return reg

    @kernel
    def qalloc(
        slot_indices: list[int] | IList[int, Any],
        theta: float = 0.0,
        phi: float = 0.0,
        lam: float = 0.0,
    ) -> IList[LogicalQubit, Any]:
        """
        Allocates a list of logical qubits with the same theta, phi, lam angles.
        """

        def _inner(slot_index: int):
            return qalloc_slot(slot_index, theta, phi, lam)

        return ilist.map(_inner, slot_indices)

    return qalloc, qalloc_slot

In [ ]:
# qalloc is used 
qalloc, qalloc_slot = steane_slot_allocator()

N = TypeVar("N")

slots: IList([IList([(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 0, 3), (0, 0, 4), (0, 0, 5), (0, 0, 6)]), IList([(0, 4, 0), (0, 4, 1), (0, 4, 2), (0, 4, 3), (0, 4, 4), (0, 4, 5), (0, 4, 6)]), IList([(0, 8, 0), (0, 8, 1), (0, 8, 2), (0, 8, 3), (0, 8, 4), (0, 8, 5), (0, 8, 6)]), IList([(0, 12, 0), (0, 12, 1), (0, 12, 2), (0, 12, 3), (0, 12, 4), (0, 12, 5), (0, 12, 6)]), IList([(0, 16, 0), (0, 16, 1), (0, 16, 2), (0, 16, 3), (0, 16, 4), (0, 16, 5), (0, 16, 6)]), IList([(0, 2, 0), (0, 2, 1), (0, 2, 2), (0, 2, 3), (0, 2, 4), (0, 2, 5), (0, 2, 6)]), IList([(0, 6, 0), (0, 6, 1), (0, 6, 2), (0, 6, 3), (0, 6, 4), (0, 6, 5), (0, 6, 6)]), IList([(0, 10, 0), (0, 10, 1), (0, 10, 2), (0, 10, 3), (0, 10, 4), (0, 10, 5), (0, 10, 6)]), IList([(0, 14, 0), (0, 14, 1), (0, 14, 2), (0, 14, 3), (0, 14, 4), (0, 14, 5), (0, 14, 6)]), IList([(0, 20, 0), (0, 20, 1), (0, 20, 2), (0, 20, 3), (0, 20, 4), (0, 20, 5), (0, 20, 6)])])


In [9]:
# However, because our gates operate on Qubits, not LogicalQubits, we need some way of obtaining a list of Qubits to do gate operations.
@kernel
def flat(
    reg: ilist.IList[LogicalQubit, Any],
) -> ilist.IList[Qubit, Any]:
    """Flatten a logical register into a single list of physical qubits"""

    def _inner(cumulant, ele):
        return cumulant + ele

    return ilist.foldl(_inner, reg, ilist.IList([]))


# Gadgets

In [10]:
N = TypeVar("N")

In [ ]:
# TODO: think of a good gadget use case/example that's not as trivial as the CX?
@kernel
def cx(controls: ilist.IList[LogicalQubit, N], targets: ilist.IList[LogicalQubit, N]):
    """Efficient broadcasted cx gate over steane logical qubits"""
    squin.broadcast.cx(flat(controls), flat(targets))